# Exploratory Data Analysis (EDA) Notebook

A comprehensive, well-structured exploratory data analysis pipeline for PISA-style educational assessment data.

## Overview
This notebook performs:
- Data loading with robust error handling
- Missing value analysis and visualization
- Target variable distribution analysis
- Feature correlation analysis
- Demographic breakdowns (gender, country)
- Wellbeing and mental health composite scoring

**Usage:** Update the `X_PATH` and `Y_PATH` variables in Section 2, then execute cells sequentially.

---
## 1. Setup & Imports

In [ ]:
"""Import required libraries and configure visualization settings."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Visualization configuration
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

# Display library versions for reproducibility
print(f"pandas: {pd.__version__}, numpy: {np.__version__}, seaborn: {sns.__version__}")

---
## 2. Configuration

Define file paths for your feature and target datasets.

In [ ]:
# ============ USER CONFIGURATION ============
X_PATH = r'C:/Users/yanis/Downloads/x_train.csv'  # Features file path
Y_PATH = r'C:/Users/yanis/Downloads/y_train.csv'  # Target file path
# ============================================

# Validate file paths
for label, path in [('X_PATH', X_PATH), ('Y_PATH', Y_PATH)]:
    if not os.path.exists(path):
        print(f"Warning: {label} not found at '{path}'")

---
## 3. Data Loading

Load datasets with error handling and fallback to synthetic demo data.

In [ ]:
def load_csv_safe(filepath: str, nrows: int = None) -> pd.DataFrame:
    """Load CSV file with existence check.
    
    Args:
        filepath: Path to CSV file
        nrows: Number of rows to read (None for all)
    
    Returns:
        DataFrame containing CSV data
    
    Raises:
        FileNotFoundError: If file does not exist
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    return pd.read_csv(filepath, nrows=nrows)


def generate_demo_data(n_samples: int = 100) -> tuple:
    """Generate synthetic demo data for testing.
    
    Args:
        n_samples: Number of samples to generate
    
    Returns:
        Tuple of (X, y) DataFrames
    """
    np.random.seed(42)
    
    X_demo = pd.DataFrame({
        'Unnamed: 0': range(1, n_samples + 1),
        'Year': np.random.choice([2015, 2018, 2022], size=n_samples),
        'CNT': np.random.choice(['FRA', 'USA', 'BRA', 'NLD', 'ESP'], size=n_samples),
        'ST004D01T': np.random.choice([1, 2], size=n_samples),
        'ST038': np.random.randint(0, 5, size=n_samples),
        'ST034': np.random.randint(0, 5, size=n_samples),
        'WB155': np.random.randint(0, 11, size=n_samples),
        'reading_q15_average_score': np.random.normal(100, 20, n_samples),
        'math_q16_average_score': np.random.normal(90, 25, n_samples),
    })
    
    y_demo = pd.DataFrame({
        'Unnamed: 0': X_demo['Unnamed: 0'],
        'MathScore': np.abs(np.random.normal(100, 50, n_samples))
    })
    
    return X_demo, y_demo


# Attempt to load data with fallback
try:
    X = load_csv_safe(X_PATH)
    y = load_csv_safe(Y_PATH)
    print("✓ Files loaded successfully.")
except FileNotFoundError as e:
    print(f"✗ {e}")
    print("→ Creating synthetic demo data...")
    X, y = generate_demo_data()

---
## 4. Data Overview

Inspect dataset dimensions, column types, and summary statistics.

In [ ]:
# Dataset dimensions
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")

# Preview data
print("\n--- Features Preview ---")
display(X.head())

print("\n--- Target Preview ---")
display(y.head())

# Summary statistics (first 10 columns)
print("\n--- Summary Statistics (First 10 Columns) ---")
display(X.describe(include='all').T.head(10))

---
## 5. Merge Features and Target

Combine feature and target datasets into a single DataFrame.

In [ ]:
def merge_features_target(X: pd.DataFrame, y: pd.DataFrame, 
                          key_col: str = 'Unnamed: 0',
                          target_col: str = 'MathScore') -> pd.DataFrame:
    """Merge feature and target DataFrames.
    
    Args:
        X: Features DataFrame
        y: Target DataFrame
        key_col: Column name to use as merge key
        target_col: Target column name
    
    Returns:
        Merged DataFrame
    """
    # Merge on key column if present in both DataFrames
    if key_col in X.columns and key_col in y.columns:
        merged = X.merge(y[[key_col, target_col]], on=key_col, how='left')
    elif len(X) == len(y):
        # Fallback: align by index if same length
        merged = X.copy()
        merged[target_col] = y[target_col].values if target_col in y.columns else y.iloc[:, -1].values
    else:
        raise ValueError("Cannot merge: no common key and different row counts.")
    
    return merged


# Execute merge
data = merge_features_target(X, y)
print(f"Merged dataset shape: {data.shape}")
display(data.head())

---
## 6. Missing Value Analysis

Quantify and visualize missing data patterns.

In [ ]:
def analyze_missing_values(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Calculate missing value statistics for each column.
    
    Args:
        df: Input DataFrame
        top_n: Number of top columns to return
    
    Returns:
        DataFrame with missing counts and percentages
    """
    missing_count = df.isnull().sum().sort_values(ascending=False)
    missing_pct = (missing_count / len(df)) * 100
    
    summary = pd.DataFrame({
        'missing_count': missing_count,
        'missing_percent': missing_pct
    })
    
    return summary.head(top_n)


# Generate missing value summary
missing_summary = analyze_missing_values(data, top_n=20)
display(missing_summary)

# Visualize columns with missing values
top_missing = missing_summary[missing_summary['missing_count'] > 0].head(30)

if not top_missing.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(x=top_missing.index, y=top_missing['missing_percent'], ax=ax, color='steelblue')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_ylabel('Percent Missing')
    ax.set_title('Top Columns by Missing Value Percentage')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected in top columns.")

---
## 7. Target Variable Distribution

Analyze the distribution of `MathScore`, handling zero values explicitly.

In [ ]:
# Verify target column exists
TARGET_COL = 'MathScore'
if TARGET_COL not in data.columns:
    raise KeyError(f"'{TARGET_COL}' column not found in merged data.")

# Summary statistics
print("Target Variable Summary:")
display(data[TARGET_COL].describe())

# Filter non-zero scores for distribution plot
nonzero_scores = data.loc[data[TARGET_COL] > 0, TARGET_COL]

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(nonzero_scores, kde=True, bins=40, color='royalblue', ax=ax)
ax.set_title('Distribution of MathScore (Excluding Zeros)')
ax.set_xlabel('MathScore')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

# Report zero-score proportion
zero_fraction = (data[TARGET_COL] == 0).mean()
print(f"\nFraction of zero MathScore values: {zero_fraction:.3f}")

---
## 8. Feature Type Classification

Identify numerical and categorical columns.

In [ ]:
# Classify columns by data type
numerical_cols = data.select_dtypes(include=['number']).columns.tolist()
categorical_cols = data.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns:   {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nSample categorical columns: {categorical_cols[:10]}")

---
## 9. Feature Correlation Analysis

Compute and visualize correlations between numerical features and the target.

In [ ]:
# Calculate absolute correlations with target
correlations = (
    data[numerical_cols]
    .corrwith(data[TARGET_COL])
    .dropna()
    .abs()
    .sort_values(ascending=False)
)

print("Top 20 Features by Correlation with MathScore:")
display(correlations.head(20))

# Heatmap of top correlated features
top_features = correlations.head(20).index.tolist()

if len(top_features) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr_matrix = data[top_features].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, ax=ax, square=True)
    ax.set_title('Correlation Matrix - Top Features')
    plt.tight_layout()
    plt.show()

---
## 10. Gender Analysis

Analyze score distributions by gender (`ST004D01T`).

In [ ]:
GENDER_COL = 'ST004D01T'

if GENDER_COL in data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gender distribution
    sns.countplot(x=GENDER_COL, data=data, palette='magma', ax=axes[0])
    axes[0].set_title(f'Count by Gender ({GENDER_COL})')
    axes[0].set_xlabel('Gender')
    axes[0].set_ylabel('Count')
    
    # Score distribution by gender (all data)
    sns.violinplot(x=GENDER_COL, y=TARGET_COL, data=data, 
                   palette='pastel', inner='quartile', ax=axes[1])
    axes[1].set_title('MathScore by Gender (All Data)')
    axes[1].set_xlabel('Gender')
    axes[1].set_ylabel('MathScore')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Gender column '{GENDER_COL}' not found in dataset.")

In [ ]:
# Score distribution by gender (excluding zeros)
if GENDER_COL in data.columns:
    nonzero_data = data[data[TARGET_COL] > 0]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.violinplot(x=GENDER_COL, y=TARGET_COL, data=nonzero_data, 
                   palette='pastel', inner='quartile', ax=ax)
    ax.set_title('MathScore Distribution by Gender (Excluding Zero Scores)')
    ax.set_xlabel('Gender')
    ax.set_ylabel('MathScore')
    plt.tight_layout()
    plt.show()

---
## 11. Country-Level Comparison

Compare score distributions across countries.

In [ ]:
COUNTRY_COL = 'CNT'

if COUNTRY_COL in data.columns:
    # Filter non-zero scores
    data_filtered = data[data[TARGET_COL] > 0].copy()
    
    # Rank countries by mean score
    country_means = data_filtered.groupby(COUNTRY_COL)[TARGET_COL].mean().sort_values(ascending=False)
    sorted_countries = country_means.index.tolist()
    
    # Select top, middle, and bottom countries for comparison
    if len(sorted_countries) >= 9:
        n = len(sorted_countries)
        top_3 = sorted_countries[:3]
        middle_3 = sorted_countries[n // 2 - 1 : n // 2 + 2]
        bottom_3 = sorted_countries[-3:]
        selected_countries = top_3 + middle_3 + bottom_3
        
        subset = data_filtered[data_filtered[COUNTRY_COL].isin(selected_countries)]
        
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.boxplot(x=COUNTRY_COL, y=TARGET_COL, data=subset, 
                    order=selected_countries, palette='viridis', ax=ax)
        ax.set_title('MathScore Distribution: Top, Middle, and Bottom Countries')
        ax.set_xlabel('Country')
        ax.set_ylabel('MathScore')
        plt.tight_layout()
        plt.show()
    else:
        print("Insufficient countries for top/middle/bottom comparison.")
else:
    print(f"Country column '{COUNTRY_COL}' not found in dataset.")

In [ ]:
# Full country comparison bar plot
if COUNTRY_COL in data.columns:
    country_order = (
        data[data[TARGET_COL] > 0]
        .groupby(COUNTRY_COL)[TARGET_COL]
        .mean()
        .sort_values(ascending=False)
        .index
    )
    
    fig, ax = plt.subplots(figsize=(20, 6))
    sns.barplot(x=COUNTRY_COL, y=TARGET_COL, data=data, 
                order=country_order, palette='viridis', ax=ax)
    ax.set_title('Mean MathScore by Country (Ordered by Performance)')
    ax.set_xlabel('Country')
    ax.set_ylabel('Mean MathScore')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    plt.tight_layout()
    plt.show()

---
## 12. Wellbeing & Mental Health Analysis

Analyze relationships between wellbeing indicators and academic performance.

In [ ]:
# Define wellbeing-related columns
WELLBEING_COLS = ['ST034', 'ST038', 'ST266', 'ST354', 'WB153', 'WB154', 'WB155', 'WB178']

# Filter to available columns
available_wellbeing_cols = [col for col in WELLBEING_COLS if col in data.columns]
print(f"Available wellbeing columns: {available_wellbeing_cols}")

# Create subset for wellbeing analysis
if available_wellbeing_cols:
    data_wellbeing = data[available_wellbeing_cols + [TARGET_COL]].copy()

In [ ]:
# Wellbeing correlation matrix
if available_wellbeing_cols:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    corr = data_wellbeing.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-0.5, vmax=0.5, center=0, ax=ax, square=True)
    ax.set_title('Correlation Matrix: Wellbeing Variables vs MathScore')
    plt.tight_layout()
    plt.show()

### Bullying Impact Analysis

In [ ]:
BULLYING_COL = 'ST038'

if BULLYING_COL in data_wellbeing.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=BULLYING_COL, y=TARGET_COL, data=data_wellbeing, 
                palette='Reds', ax=ax)
    ax.set_title('Mean MathScore by Bullying Exposure Level')
    ax.set_xlabel(f'Bullying Exposure ({BULLYING_COL})')
    ax.set_ylabel('Mean MathScore')
    ax.set_ylim(15, 200)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()

### Life Satisfaction Analysis

In [ ]:
LIFE_SAT_COL = 'WB155'

if LIFE_SAT_COL in data_wellbeing.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x=LIFE_SAT_COL, y=TARGET_COL, data=data_wellbeing, 
                palette='viridis', ax=ax)
    ax.set_title('Mean MathScore by Life Satisfaction Level')
    ax.set_xlabel(f'Life Satisfaction ({LIFE_SAT_COL}, 0-10 scale)')
    ax.set_ylabel('Mean MathScore')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()

---
## 13. Mental Health Composite Score

Construct a standardized composite mental health index from wellbeing variables.

### Methodology

Each item is first standardized:

$$
Z_{ik} = \frac{S_{ik} - \mu_k}{\sigma_k}
$$

The composite mental health score is computed as:

$$
\text{MentalHealth}_i = \frac{1}{N} \sum_{k=1}^{N} Z_{ik}
$$

where negative indicators are reversed (multiplied by -1) before averaging.

In [ ]:
# Define indicator types
POSITIVE_INDICATORS = ['ST034', 'WB153', 'WB155', 'WB178', 'ST354']
NEGATIVE_INDICATORS = ['ST038', 'ST266', 'WB154']

# Filter to available features
positive_cols = [c for c in POSITIVE_INDICATORS if c in data.columns]
negative_cols = [c for c in NEGATIVE_INDICATORS if c in data.columns]
all_mh_features = positive_cols + negative_cols

print(f"Positive indicators found: {positive_cols}")
print(f"Negative indicators found: {negative_cols}")

if all_mh_features:
    # Prepare data subset
    df_mh = data[all_mh_features + [TARGET_COL]].copy()
    df_mh = df_mh.dropna(subset=all_mh_features, thresh=len(all_mh_features) - 2)
    
    # Standardize features
    scaler = StandardScaler()
    standardized_cols = []
    
    for col in positive_cols:
        col_std = f'{col}_std'
        df_mh[col_std] = scaler.fit_transform(df_mh[[col]])
        standardized_cols.append(col_std)
    
    for col in negative_cols:
        col_std = f'{col}_std'
        df_mh[col_std] = scaler.fit_transform(df_mh[[col]]) * -1  # Reverse negative indicators
        standardized_cols.append(col_std)
    
    # Compute composite score
    df_mh['MentalHealth_Score'] = df_mh[standardized_cols].mean(axis=1)
    
    # Create quintile groups
    quintile_labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
    df_mh['MH_Quintile'] = pd.qcut(df_mh['MentalHealth_Score'], q=5, labels=quintile_labels)
    
    print(f"\nMental Health Score computed for {len(df_mh):,} observations.")
else:
    print("No wellbeing features available for mental health index computation.")

In [ ]:
# Visualize MathScore by Mental Health Quintile
if all_mh_features:
    stats = (
        df_mh.groupby('MH_Quintile')[TARGET_COL]
        .agg(['mean', 'median'])
        .reindex(quintile_labels)
    )
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(stats.index, stats['mean'], marker='o', linewidth=2, 
            label='Mean', color='steelblue')
    ax.plot(stats.index, stats['median'], marker='s', linewidth=2, 
            label='Median', color='coral')
    ax.set_xlabel('Mental Health Quintile')
    ax.set_ylabel('MathScore')
    ax.set_title('Mean and Median MathScore by Mental Health Quintile')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# Fine-grained analysis: MathScore by Mental Health percentile bins
if all_mh_features:
    df_mh['MH_Percentile'] = pd.qcut(df_mh['MentalHealth_Score'], q=20, duplicates='drop')
    percentile_means = df_mh.groupby('MH_Percentile')[TARGET_COL].mean()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    percentile_means.plot(kind='line', marker='o', ax=ax, color='darkgreen')
    ax.set_title('Mean MathScore by Mental Health Percentile (20 bins)')
    ax.set_xlabel('Mental Health Percentile Bin')
    ax.set_ylabel('Mean MathScore')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()